In [ ]:
# Cell 0b: Install dependencies (run once)
# Ref: requirements.txt

import subprocess, sys

def install_requirements():
    """Install dependencies from requirements.txt if available."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q",
             "-r", "requirements.txt"],
            stdout=subprocess.DEVNULL
        )
        print("Dependencies installed successfully.")
    except FileNotFoundError:
        print("requirements.txt not found. Continuing with available packages.")
    except subprocess.CalledProcessError:
        print("Some packages may not have installed. Notebook will use Simulation Mode if needed.")

install_requirements()

In [ ]:
# Cell 0c: Core imports, API key detection, and mode selection
# Ref: Strategy §2.3 — API Key Detection Flow

import os
import sys
import json
from datetime import datetime

# Ensure chapter01 package is importable from repo root
if os.path.dirname(os.path.abspath('')) not in sys.path:
    sys.path.insert(0, os.path.dirname(os.path.abspath('')))

from chapter01.utils import (
    log_info, log_success, log_error, log_warning,
    graceful_fallback, detect_api_key, get_provider,
    simulation_mode_banner, live_mode_banner
)
from chapter01.mock_llm import MockLLM, MockResponse

# ── Multi-provider API Key Detection ──
# LLM_PROVIDER env var controls provider: openai | anthropic | google | auto
api_key, MODE = detect_api_key()
PROVIDER = get_provider()

if MODE == "LIVE":
    try:
        sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), '..'))
        from supporting.llm_provider import get_client, chat_completion, PROVIDER_MODELS, print_provider_banner
        client = get_client(provider=PROVIDER, api_key=api_key)
        default_model = PROVIDER_MODELS[PROVIDER]["default"]
        print_provider_banner(PROVIDER, MODE, default_model)

        def llm_chat(messages, temperature=0.7):
            """Multi-provider chat wrapper (OpenAI / Anthropic / Google)."""
            response_text = chat_completion(
                client, PROVIDER, messages,
                model=default_model,
                temperature=temperature,
            )
            return MockResponse(content=response_text, model=default_model)
    except ImportError as e:
        log_warning(f"Provider package not available: {e}. Falling back to Simulation Mode.")
        MODE = "SIMULATION"
    except Exception as e:
        log_warning(f"Provider init failed: {e}. Falling back to Simulation Mode.")
        MODE = "SIMULATION"

if MODE == "SIMULATION":
    llm = MockLLM(simulate_latency=True, failure_rate=0.0)
    simulation_mode_banner()

    def llm_chat(messages, temperature=0.7):
        """Simulation Mode chat wrapper."""
        return llm.chat(messages, temperature=temperature)

# ── JSON Response Parser (handles both Mock and Live LLM responses) ──
def parse_json_response(content):
    """Extract and parse JSON from an LLM response.
    
    Handles: pure JSON, markdown code blocks, prose-wrapped JSON.
    Author: Imran Ahmad
    """
    import re
    text = content.strip()
    # Try direct parse first
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        pass
    # Try extracting from markdown code block
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        try:
            return json.loads(match.group(1).strip())
        except (json.JSONDecodeError, ValueError):
            pass
    # Try finding first { or [ and matching to last } or ]
    for start_char, end_char in [('{', '}'), ('[', ']')]:
        start = text.find(start_char)
        end = text.rfind(end_char)
        if start != -1 and end > start:
            try:
                return json.loads(text[start:end+1])
            except (json.JSONDecodeError, ValueError):
                pass
    raise ValueError(f"Could not extract JSON from response: {text[:200]}")

log_success(f"Notebook initialized in {MODE} mode — Provider: {PROVIDER}")

In [ ]:
# Cell 1b: Timeline visualization — four eras of AI agent evolution
# Ref: §1.1 — Introducing Agents

log_info("Rendering AI agent evolution timeline (§1.1)...")

timeline = [
    ("1970s-1980s", "Rule-Based Expert Systems",
     "MYCIN, logic-based inference engines. Deterministic but brittle."),
    ("1990s",       "Classical Machine Learning",
     "Decision trees, SVMs. Pattern recognition but task-specific and stateless."),
    ("2010s",       "Deep Learning Revolution",
     "Speech, image, translation at human-level. Largely reactive input-output."),
    ("2020s+",      "LLMs & Autonomous Agents",
     "Emergent reasoning, tool use, memory. Frameworks: LangGraph, CrewAI, AutoGen."),
]

print("\n" + "=" * 70)
print("  THE EVOLUTION OF AI AGENTS")
print("=" * 70)

for i, (era, title, desc) in enumerate(timeline):
    connector = "│" if i < len(timeline) - 1 else " "
    print(f"\n  {era:<14}  ── {title}")
    print(f"  {'':14}     {desc}")
    if i < len(timeline) - 1:
        print(f"  {'':14}     │")
        print(f"  {'':14}     ▼")

print("\n" + "=" * 70)

# Customer support progression (§1.1)
print("\n  CUSTOMER SUPPORT PROGRESSION (§1.1):")
print("  ─────────────────────────────────────")
support_eras = [
    ("2010", "Static FAQ scripts — predetermined responses, human intervention for deviations"),
    ("2018", "ML-based ticket routing — categorize and assign, still human resolution"),
    ("2025", "Multi-agent systems — 70-85% autonomous resolution, LLMs + live knowledge bases"),
]
for year, desc in support_eras:
    print(f"  {year}:  {desc}")

log_success("Timeline rendered successfully.")

In [ ]:
# Cell 1: Cognitive Loop — Phase 1: Perception


@graceful_fallback(
    fallback_value={"message": "fallback_input", "sentiment": "neutral", "user_id": "UNKNOWN Respond with valid JSON only, no additional text."},
    section_ref="§1.2.1 Perception"
)
def perceive_input(user_message):
    """Capture and structure raw user input into a perception dict.

    Mirrors the chapter's perceive_input() pseudocode:
    captures message, timestamp, user_id, session_state, and sentiment.

    Author: Imran Ahmad
    """
    response = llm_chat([
        {"role": "system", "content": "You are a perception module. Analyze the input and return structured perception data as valid JSON only, no additional text."},
        {"role": "user", "content": f"Perceive and capture sentiment for this customer input: {user_message}"}
    ])
    result = parse_json_response(response.content)
    return result

# Run perception on the chapter's billing scenario
perception = perceive_input("I need help with my billing")
print("\nPerception Output:")
print(json.dumps(perception, indent=2))

In [ ]:
# Cell 2c: Cognitive Loop — Phase 2: Reasoning
# Ref: §1.2.1 — "Reasoning follows by contextualizing perceived information"

@graceful_fallback(
    fallback_value={"intent": "unknown", "priority": "medium", "confidence": 0.0},
    section_ref="§1.2.1 Reasoning"
)
def reason_about_intent(perception_data):
    """Classify intent and priority from perception data.

    Mirrors the chapter's reason_about_intent() pseudocode:
    classifies intent, determines priority based on sentiment and history.

    Author: Imran Ahmad
    """
    response = llm_chat([
        {"role": "system", "content": "You are a reasoning module. Classify the intent and priority. Respond with valid JSON only, no additional text."},
        {"role": "user", "content": f"Reason about the intent and classify priority for: {json.dumps(perception_data)}"}
    ])
    result = parse_json_response(response.content)
    return result

# Run reasoning on perception output
reasoning = reason_about_intent(perception)
print("\nReasoning Output:")
print(json.dumps(reasoning, indent=2))

In [ ]:
# Cell 2d: Cognitive Loop — Phase 3: Planning
# Ref: §1.2.1 — "Planning orchestrates insights into a coherent sequence"

@graceful_fallback(
    fallback_value=["log_request", "escalate_to_human"],
    section_ref="§1.2.1 Planning"
)
def create_action_plan(reasoning_result):
    """Decompose the intent into an ordered list of action steps.

    Mirrors the chapter's create_action_plan() pseudocode:
    for billing_issue → fetch_account, analyze_history, generate_explanation, offer_resolution.

    Author: Imran Ahmad
    """
    response = llm_chat([
        {"role": "system", "content": "You are a planning module. Create a step-by-step action plan. Respond with valid JSON only, no additional text."},
        {"role": "user", "content": f"Create an action plan to decompose steps for billing issue: {json.dumps(reasoning_result)}"}
    ])
    result = parse_json_response(response.content)
    return result

# Run planning on reasoning output
action_plan = create_action_plan(reasoning)
print("\nAction Plan:")
for i, step in enumerate(action_plan, 1):
    print(f"  Step {i}: {step}")